In [ ]:
import numpy as np

npy_path = '/data/coding/upload-data/data/DENSE/test/events/voxels/event_tensor_0000000000.npy'

data = np.load(npy_path)
print(data.shape)

#### Process MVSEC(hdf5)

* `uint8`: `<HDF5 dataset "image_raw": shape (11937, 260, 346), type "|u1">`
* `float64`: `<HDF5 dataset "image_raw_ts": shape (11937,), type "<f8">`
* `float32`: `<HDF5 dataset "depth_image_raw": shape (5134, 260, 346), type "<f4">`
* `float64`: `<HDF5 dataset "events": shape (105874257, 4), type "<f8">`


In [1]:
from tqdm import tqdm

import json
import numpy as np

import h5py
import os
import cv2

In [17]:
def map_depth_to_image(image_ts, depth_ts, output_dir):
    """
    Map depth timestamps to the nearest previous image timestamps.
    
    Args:
        image_ts (numpy.ndarray): timestamps of each image.
        depth_ts (numpy.ndarray): timestamps of each depth.
        output_dir (str): Directory to save the output JSON files.
        
    Output:
        Saves two JSON files:
        - depth_to_image.json: Mapping from depth indices to nearest image indices.
    """
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    # Create mappings
    depth_to_image = {}

    # Compute nearest previous image for each depth
    for depth_idx, depth_time in enumerate(depth_ts):
        nearest_image_idx = None
        for idx, ts in enumerate(image_ts):
            if ts > depth_time:
                break
            nearest_image_idx = idx
        nearest_image_idx = nearest_image_idx if nearest_image_idx is not None else len(image_ts) - 1
        
        item = {}
        item['nearest_img_idx'] = nearest_image_idx
        item['depth_ts'] = depth_time
        item['img_ts'] = image_ts[nearest_image_idx]
        item['diff'] = abs(item['depth_ts'] - item['img_ts'])
        
        depth_to_image[depth_idx] = item

    # Save mappings to JSON files
    with open(f"{output_dir}/depth_to_image.json", "w") as f:
        json.dump(depth_to_image, f, indent=4)

    print(f"Mapping saved to {output_dir}/depth_to_image.json")

In [20]:
def map_depth_to_events(depth_ts, events, output_dir):
    """
    Maps each depth map timestamp to a range of events within a specific time window.

    Parameters:
        depth_ts (numpy.ndarray): timestamps of each depth map.
        events (numpy.ndarray): each row represents an event as (x, y, timestamp, p).
        output_dir (str): Directory to save the output JSON file.

    Output:
        A JSON file saved to output_dir containing the correspondence between depth maps and events.
    """
    
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    # Initialize result dictionary
    depth_to_events_mapping = {}

    # Start index for events
    current_event_index = 0

    # Iterate through each depth timestamp
    for i, depth_time in enumerate(depth_ts):
        # Move the start index forward to skip past events older than 50ms before the depth timestamp
        while current_event_index < len(events) and events[current_event_index, 2] < depth_time - 0.05:
            current_event_index += 1

        # Find the closest event index before the depth timestamp
        end_event_index = current_event_index
        while end_event_index < len(events) and events[end_event_index, 2] <= depth_time:
            end_event_index += 1

        if current_event_index < len(events) and end_event_index > current_event_index:
            # Save the mapping
            depth_to_events_mapping[i] = {
                "depth_timestamp": float(depth_time),
                "start_event_index": int(current_event_index),
                "end_event_index": int(end_event_index - 1),
                "start_event_timestamp": float(events[current_event_index, 2]),
                "end_event_timestamp": float(events[end_event_index - 1, 2])
            }

    # Save the mapping to a JSON file
    output_file = os.path.join(output_dir, "depth_to_events_mapping.json")
    with open(output_file, "w") as f:
        json.dump(depth_to_events_mapping, f, indent=4)

    print(f"Mapping saved to {output_file}")

In [11]:
def check_hdf5_data_struct(hdf5_file):
    def prt_name(name):
        print(name)

    hdf5_file.visit(prt_name)

def save_images2directory(images, output_dir):
    """Extract and save the grayscale image to the specified directory"""
    
    for i, img in enumerate(tqdm(images, desc="Saving images", unit="img")):
        # Construct the output file path
        output_path = os.path.join(output_dir, f"{i:05}.png")

        # Save the image using OpenCV (as grayscale)
        cv2.imwrite(output_path, img)

In [12]:
data_hdf5_path = r"F:\MVSEC\mvsec-hdf5\outdoor_day1_data.hdf5"
gt_hdf5_path = r"F:\MVSEC\mvsec-hdf5\outdoor_day1_gt.hdf5"

if os.path.isfile(data_hdf5_path):
    print(f"Data file {data_hdf5_path} exists")
if os.path.isfile(gt_hdf5_path):
    print(f"GT file {gt_hdf5_path} exists")

data = h5py.File(data_hdf5_path)
gt = h5py.File(gt_hdf5_path)

Data file F:\MVSEC\mvsec-hdf5\outdoor_day1_data.hdf5 exists
GT file F:\MVSEC\mvsec-hdf5\outdoor_day1_gt.hdf5 exists


In [ ]:
# print(f"Data Struct of {data_hdf5_path}")
# check_hdf5_data_struct(data)
# print(f"Data Struct of {gt_hdf5_path}")
# check_hdf5_data_struct(gt)

In [13]:
images = data['davis']['left']['image_raw']
image_ts = data['davis']['left']['image_raw_ts']
events = data['davis']['left']['events'] # x, y, t, p
depth_image_raw = gt['davis']['left']['depth_image_raw']
depth_image_raw_ts = gt['davis']['left']['depth_image_raw_ts']

In [ ]:
# Check if the timestamps of the events are in increasing order
events = np.array(events)
pre_time = 0
flag = True

for event in events:
    if event[2] < pre_time:
        flag = False
        break
    pre_time = event[2]

if flag:
    print("All event timestamps are in increasing order.")
else:
    print("Event timestamps are not in increasing order.")
    print("Sorting the events by timestamp in ascending order...")
    sorted_indices = np.argsort(events[:, 2])
    sorted_arr = events[sorted_indices]
    print("Sorting is complete. Events are now ordered by timestamp.")

# Check if the timestamps of the depth are in increasing order
if np.all(depth_image_raw_ts[:-1] <= depth_image_raw_ts[1:]):
    print("All depth timestamps are in increasing order.")
else:
    pass
# Check if the timestamps of the images are in increasing order
if np.all(image_ts[:-1] <= image_ts[1:]):
    print("All image timestamps are in increasing order.")
else:
    pass

In [14]:
map_index_dir = r'F:\MVSEC\mvsec-hdf5\test\data'
map_depth_to_image(image_ts, depth_image_raw_ts, map_index_dir)

Mappings saved:
 - image_to_depth.json: F:\MVSEC\mvsec-hdf5\test\data/image_to_depth.json
 - depth_to_image.json: F:\MVSEC\mvsec-hdf5\test\data/depth_to_image.json


The timestamp of the event is increasing


In [15]:
print(image_ts[100] - image_ts[100 + 45])

-0.9854640960693359


In [16]:
print(type(images), type(image_ts))
print(images)
print(image_ts)
print(events)
print(image_raw_event_inds)

print(depths)
print(rect_depths)

<class 'h5py._hl.dataset.Dataset'> <class 'h5py._hl.dataset.Dataset'>
<HDF5 dataset "image_raw": shape (11937, 260, 346), type "|u1">
<HDF5 dataset "image_raw_ts": shape (11937,), type "<f8">
<HDF5 dataset "events": shape (105874257, 4), type "<f8">
<HDF5 dataset "image_raw_event_inds": shape (11937,), type "<i8">
<HDF5 dataset "depth_image_raw": shape (5134, 260, 346), type "<f4">
<HDF5 dataset "depth_image_rect": shape (5134, 260, 346), type "<f4">


In [18]:
print(type(image_ts))

<class 'h5py._hl.dataset.Dataset'>


#### Generate DENSE split file

In [2]:
import os

In [4]:
def get_files_in_dir(dir, file_types=['png', 'npy']):
    files = os.listdir(dir)
    files = [path for path in files if path.endswith((".png", ".npy"))]
    files.sort()
    files = [os.path.join(dir, path) for path in files]
    return files

In [18]:
data_root = '/data/coding/upload-data/data/DENSE/val'
towns = os.listdir(data_root)

rgbs, depths, events = [], [], []

for town in towns:
    rgb_dir = os.path.join(data_root, town, 'rgb', 'frames')
    depth_dir = os.path.join(data_root, town, 'depth', 'data')
    event_dir = os.path.join(data_root, town, 'events', 'voxels')
    
    rgb_name = get_files_in_dir(rgb_dir)
    depth_name = get_files_in_dir(depth_dir)
    event_name = get_files_in_dir(event_dir)
    
    rgbs = rgbs + rgb_name
    depths = depths + depth_name
    events = events + event_name

split_name = '/data/coding/upload-data/data/DENSE/val.txt'
lines = []

assert len(rgbs) == len(depths) and len(rgbs) == len(events)

for i in range(len(rgbs)):
    lines.append(rgbs[i] + ' ' + depths[i] + ' ' + events[i] + '\n')

with open(split_name, 'w') as f:
    f.writelines(lines)

In [6]:
data_root = '/data/coding/upload-data/data/DENSE/test'

rgb_dir = os.path.join(data_root, 'rgb', 'frames')
depth_dir = os.path.join(data_root, 'depth', 'data')
event_dir = os.path.join(data_root, 'events', 'voxels')

rgb_name = get_files_in_dir(rgb_dir)
depth_name = get_files_in_dir(depth_dir)
event_name = get_files_in_dir(event_dir)

split_name = '/data/coding/upload-data/data/DENSE/test.txt'
lines = []

assert len(rgb_name) == len(depth_name) and len(rgb_name) == len(event_name)

for i in range(len(rgb_name)):
    lines.append(rgb_name[i] + ' ' + depth_name[i] + ' ' + event_name[i] + '\n')

with open(split_name, 'w') as f:
    f.writelines(lines)

#### Delete redundant files in dense

In [6]:
import os
import shutil

In [ ]:
data_root = '/data/coding/upload-data/data/DENSE/train'

towns = os.listdir(data_root)

for town in towns:
    town_path = os.path.join(data_root, town)
    depth_frames = os.path.join(town_path, 'depth', 'frames')
    event_frames = os.path.join(town_path, 'events', 'frames')
    semantic = os.path.join(town_path, 'semantic')
    
    shutil.rmtree(depth_frames)
    shutil.rmtree(event_frames)
    shutil.rmtree(semantic)

In [10]:
test_dir = '/data/coding/upload-data/data/DENSE/test'
depth_frames = os.path.join(test_dir, 'depth', 'frames')
event_frames = os.path.join(test_dir, 'events', 'frames')
semantic = os.path.join(test_dir, 'semantic')

shutil.rmtree(depth_frames)
shutil.rmtree(event_frames)
shutil.rmtree(semantic)

In [ ]:
import numpy as np
import cv2
npy_path = '/data/coding/upload-data/data/DENSE/test/events/voxels/event_tensor_0000000000.npy'

data = np.load(npy_path)
print(data.shape)
data = data.transpose(1, 2, 0)
print(data.shape)

In [ ]:
data = cv2.resize(data, (518, 686), interpolation=cv2.INTER_NEAREST)
print(data.shape)

In [51]:
def prepare_depth(depth, reg_factor, d_max):
    # Normalize depth
    # depth = np.clip(depth, 0.0, d_max)
    depth = depth / d_max
    depth = np.log(depth + 1e-6) / reg_factor + 1.0
    # depth = depth.clip(0.0, 1.0)
    return depth

In [ ]:
depth = np.array([0, -1, 80, 1000, 1200])
reg_factor, d_max = 6.2044, 1000
depth = prepare_depth(depth, reg_factor, d_max)
print(depth)
mask = (depth >= 0)
print(mask)